# Problem 4: Mixture of MNL for Early vs. Late Reservations

Customer types:

- **Type 1 / early**: `srch_booking_window >= 7`
- **Type 2 / late**: `srch_booking_window < 7`

The variable `srch_booking_window` is measured in **days**.

### For Problem 4, we estimate the same model in Problem 1 separately for early and late customers.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [2]:
DATA_PATH = "data.csv"

FEATURES = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag",
]

PARAM_NAMES = ["intercept"] + FEATURES

### Load and validate the data

In [3]:
df = pd.read_csv(DATA_PATH)

required = {"srch_id", "srch_booking_window", "booking_bool", *FEATURES}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"data.csv is missing required columns: {missing}")

# Drop rows with missing values in columns used by the model.
df = df.dropna(subset=list(required)).copy()

print(f"Rows: {len(df):,}")
print(f"Unique search queries: {df['srch_id'].nunique():,}")
df.head(30)

Rows: 153,009
Unique search queries: 8,354


,srch_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score,prop_accesibility_score,prop_log_historical_price,price_usd,promotion_flag,srch_booking_window,srch_adults_count,srch_children_count,srch_room_count,srch_saturday_night_bool,booking_bool
0,1,4,3,1,2,0,5,140,0,0,4,0,1,1,0
1,1,3,5,1,2,0,5,211,0,0,4,0,1,1,0
2,1,4,4,1,3,0,5,150,0,0,4,0,1,1,0
3,1,4,3,1,3,0,5,144,0,0,4,0,1,1,0
4,1,4,4,1,2,0,5,191,0,0,4,0,1,1,0
5,1,4,5,1,3,0,5,195,0,0,4,0,1,1,0
6,1,4,3,1,3,0,5,115,0,0,4,0,1,1,0
7,1,4,4,1,3,0,5,281,0,0,4,0,1,1,0
8,1,2,3,1,3,0,4,128,0,0,4,0,1,1,0
9,1,2,3,1,2,0,4,101,0,0,4,0,1,1,1


### Define early and late customer types

Type shares should be computed by **unique search queries**. Each search query represents one customer choice occasion.

In [4]:
def query_type_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build one row per search query so customer-type shares are not biased by
    queries that displayed more hotels.
    """
    per_query = (
        df[["srch_id", "srch_booking_window"]]
        .drop_duplicates(subset="srch_id")
        .copy()
    )
    per_query["customer_type"] = np.where(
        per_query["srch_booking_window"] >= 7,
        "type_1_early",
        "type_2_late",
    )
    return per_query

per_query = query_type_table(df)
per_query.head(10)

,srch_id,srch_booking_window,customer_type
0,1,0,type_2_late
26,75,30,type_1_early
59,101,74,type_1_early
91,218,0,type_2_late
96,236,15,type_1_early
121,325,12,type_1_early
135,419,53,type_1_early
148,1069,21,type_1_early
152,1297,1,type_2_late
168,1555,35,type_1_early


In [5]:
n_queries = per_query["srch_id"].nunique()
theta_1 = per_query["customer_type"].eq("type_1_early").sum() / n_queries
theta_2 = per_query["customer_type"].eq("type_2_late").sum() / n_queries

theta_table = pd.DataFrame({
    "type": ["type_1_early", "type_2_late"],
    "definition": ["srch_booking_window >= 7", "srch_booking_window < 7"],
    "theta": [theta_1, theta_2],
    "searches": [
        per_query["customer_type"].eq("type_1_early").sum(),
        per_query["customer_type"].eq("type_2_late").sum(),
    ],
})

theta_table

,type,definition,theta,searches
0,type_1_early,srch_booking_window >= 7,0.543093,4537
1,type_2_late,srch_booking_window < 7,0.456907,3817


In [6]:
print(f"theta_1 + theta_2 = {theta_1 + theta_2:.6f}")

theta_1 + theta_2 = 1.000000


### Attach customer type to row-level hotel data

Now we merge the customer type back onto the original hotel-level data so each type-specific MNL can be estimated on its own rows.

In [7]:
df = df.merge(per_query[["srch_id", "customer_type"]], on="srch_id", how="left")

df_early = df[df["customer_type"] == "type_1_early"].copy()
df_late = df[df["customer_type"] == "type_2_late"].copy()

summary_by_type = pd.DataFrame({
    "type": ["type_1_early", "type_2_late"],
    "searches": [df_early["srch_id"].nunique(), df_late["srch_id"].nunique()],
    "rows": [len(df_early), len(df_late)],
    "bookings": [df_early["booking_bool"].sum(), df_late["booking_bool"].sum()],
})
summary_by_type

,type,searches,rows,bookings
0,type_1_early,4537,83863,3047
1,type_2_late,3817,69146,2801


### MNL likelihood functions - reused from Problem 1

In [8]:
def compute_utilities(beta, X) -> np.ndarray:
    return beta[0] + X @ beta[1:]


def neg_log_likelihood(beta, X, y, groups) -> float:
    """
    Negative log-likelihood for MNL with no-purchase outside option.

    For each search q:
      log L_q = sum_j y_j u_j - log(1 + sum_j exp(u_j))

    If no hotel was booked, then sum_j y_j u_j = 0.
    """
    u = compute_utilities(beta, X)
    log_likelihood = 0.0

    for q in np.unique(groups):
        mask = groups == q
        u_q = u[mask]
        y_q = y[mask]

        log_likelihood += y_q @ u_q
        log_likelihood -= logsumexp(np.append(u_q, 0.0))

    return -log_likelihood

In [9]:
def prepare_groups(groups):
    _, group_index = np.unique(groups, return_inverse=True)
    return group_index


def neg_log_likelihood(beta, X, y, group_index):
    u = compute_utilities(beta, X)

    booked_utility = y @ u

    # denominator: log(1 + sum exp(u_j)) per search
    exp_u = np.exp(u)
    sum_exp_by_group = np.bincount(group_index, weights=exp_u)
    log_denom = np.log1p(sum_exp_by_group).sum()

    return -(booked_utility - log_denom)

### Estimate one MNL model per type

In [12]:
def estimate_mnl(df_type, label) -> tuple[np.ndarray, object]:
    """Estimate one MNL model for a customer type."""
    X = df_type[FEATURES].to_numpy(dtype=float)
    y = df_type["booking_bool"].to_numpy(dtype=float)
    groups = df_type["srch_id"].to_numpy()

    beta_init = np.zeros(len(PARAM_NAMES))

    result = minimize(
        neg_log_likelihood,
        beta_init,
        args=(X, y, groups),
        method="L-BFGS-B",
        options={"maxiter": 1000},
    )

    if not result.success:
        print(f"WARNING: {label} optimization did not fully converge: {result.message}")

    return result.x, result

In [13]:
beta_early, result_early = estimate_mnl(df_early, "type 1 early")
beta_late, result_late = estimate_mnl(df_late, "type 2 late")

print(f"Early model success: {result_early.success}; negative log-likelihood: {result_early.fun:.6f}")
print(f"Late model success:  {result_late.success}; negative log-likelihood: {result_late.fun:.6f}")

Early model success: True; negative log-likelihood: 11052.587387
Late model success:  True; negative log-likelihood: 9505.642630


### Compare sensitivity parameters

Positive coefficients increase utility and therefore increase booking probability, holding the displayed assortment fixed. Negative coefficients decrease utility.

In [14]:
beta_table = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "beta_type_1_early": beta_early,
    "beta_type_2_late": beta_late,
    "early_minus_late": beta_early - beta_late,
    "abs_difference": np.abs(beta_early - beta_late),
})

beta_table

,parameter,beta_type_1_early,beta_type_2_late,early_minus_late,abs_difference
0,intercept,-2.976770,-2.708354,-0.268416,0.268416
1,prop_starrating,0.442460,0.539079,-0.096618,0.096618
2,prop_review_score,0.137622,0.105939,0.031683,0.031683
3,prop_brand_bool,0.208333,0.254776,-0.046443,0.046443
4,prop_location_score,-0.015786,0.064635,-0.080421,0.080421
5,prop_accesibility_score,0.749643,0.212606,0.537036,0.537036
6,prop_log_historical_price,-0.052966,-0.016831,-0.036135,0.036135
7,price_usd,-0.005898,-0.009343,0.003445,0.003445
8,promotion_flag,0.382697,0.550249,-0.167552,0.167552


In [15]:
# Show the parameters with the largest differences first.
beta_table.sort_values("abs_difference", ascending=False)

,parameter,beta_type_1_early,beta_type_2_late,early_minus_late,abs_difference
5,prop_accesibility_score,0.749643,0.212606,0.537036,0.537036
0,intercept,-2.976770,-2.708354,-0.268416,0.268416
8,promotion_flag,0.382697,0.550249,-0.167552,0.167552
1,prop_starrating,0.442460,0.539079,-0.096618,0.096618
4,prop_location_score,-0.015786,0.064635,-0.080421,0.080421
3,prop_brand_bool,0.208333,0.254776,-0.046443,0.046443
6,prop_log_historical_price,-0.052966,-0.016831,-0.036135,0.036135
2,prop_review_score,0.137622,0.105939,0.031683,0.031683
7,price_usd,-0.005898,-0.009343,0.003445,0.003445


### Save outputs

In [17]:
beta_table.to_csv("problem4_mmnl_coefficients.csv", index=False)
theta_table.to_csv("problem4_mmnl_type_shares.csv", index=False)
summary_by_type.to_csv("problem4_mmnl_type_summary.csv", index=False)

print("Wrote:")
print("- problem4_mmnl_coefficients.csv")
print("- problem4_mmnl_type_shares.csv")
print("- problem4_mmnl_type_summary.csv")

Wrote:
- problem4_mmnl_coefficients.csv
- problem4_mmnl_type_shares.csv
- problem4_mmnl_type_summary.csv


### Final answer

- The estimated mixture weights are `theta_1 = 0.543` for early customers and `theta_2 = 0.457` for late customers.
- The early and late MNL models differ most strongly in: `prop_accesibility_score` and `promotion_flag`.
- A positive coefficient means that increasing the feature raises the hotel utility and booking probability.
- A negative price coefficient means customers prefer lower displayed prices, holding other features fixed.

- Comparison between Early and Late customer preferences:
    - Early customers weight accessibility much higher than late customers, being this by far the largest differentiator with a 0.537 difference.
    - Late customers care more about promotions and star rating in absolute and also in difference against Early ones. The difference against Early customers is larger in promotion flags.
    - All the other dimensions have less than 0.1 absolute difference.

# Problem 5 — Early vs. Late Reservations

We solve three assortment problems for each dataset `data1.csv`, `data2.csv`, `data3.csv`, and `data4.csv`:
1. Unknown arriving customer type: choose assortment `S` using the mixture model.
2. Known type 1 customer: choose assortment `S1` using the early-customer MNL.
3. Known type 2 customer: choose assortment `S2` using the late-customer MNL.

Then we compare:
- Revenue of `S` vs. `S1` under type 1.
- Revenue of `S` vs. `S2` under type 2.


In [13]:
# --- Load tables ---
beta_table = pd.read_csv("problem4_mmnl_coefficients.csv")

# Sort rows into the exact order needed by compute_utilities()
beta_table_ordered = beta_table.set_index("parameter").loc[PARAM_NAMES]

# Extract beta vectors
beta_early = beta_table_ordered["beta_type_1_early"].to_numpy(dtype=float)
beta_late = beta_table_ordered["beta_type_2_late"].to_numpy(dtype=float)

print("beta_early:")
print(beta_early)

print("\nbeta_late:")
print(beta_late)


beta_early:
[-2.97676965  0.44246037  0.13762203  0.20833311 -0.0157865   0.74964261
 -0.05296589 -0.00589757  0.38269706]

beta_late:
[-2.7083541   0.53907877  0.10593892  0.25477631  0.06463451  0.21260647
 -0.01683058 -0.00934301  0.55024875]


In [14]:
from ortools.linear_solver import pywraplp

In [15]:
PRICE_COL = "price_usd"

### Utility and preference weights

For each customer type `k`, the hotel utility is:
$$
u_{jk} = \beta_{0k} + \sum_i \beta_{ik}x_{ji}
$$

The MNL preference weight is:

$$
v_{jk} = e^{u_{jk}}
$$

In [16]:
def compute_utilities(beta, X):
    return beta[0] + X @ beta[1:]


def compute_preference_weights(df, beta):
    X = df[FEATURES].to_numpy(dtype=float)
    u = compute_utilities(beta, X)

    # Clipping prevents overflow in exp if utilities are large.
    u = np.clip(u, -50, 50)
    return np.exp(u)

### Expected revenue functions

$$
R_k(S) =
\frac{\sum_j x_j r_j v_{jk}}
{1 + \sum_j x_j v_{jk}}
$$

where:

- $x_j = 1$ if hotel $j$ is displayed, and 0 otherwise
- $r_j$ is the revenue from hotel $j$, here `price_usd`
- $v_{jk}$ is the MNL preference weight for hotel $j$ and customer type $k$

For the mixture model:

$$
R_{mix}(S) = \theta_1 R_1(S) + \theta_2 R_2(S)
$$

In [17]:
def expected_revenue_type(x, revenue, v):
    x = np.asarray(x)
    numerator = np.sum(x * revenue * v)
    denominator = 1 + np.sum(x * v)
    return numerator / denominator


def expected_revenue_mixture(x, revenue, v1, v2, theta1, theta2):
    return (
        theta1 * expected_revenue_type(x, revenue, v1)
        + theta2 * expected_revenue_type(x, revenue, v2)
    )

### Revenue-ordered assortment for known single customer type

For a single MNL model, the optimal assortment has the revenue-ordered property.

That means we can sort hotels by revenue/price in descending order and keep adding hotels while the next hotel's price is larger than the current expected revenue.

This is exact for the known-type problems `S1` and `S2`.

We will use an MILP for `S`, the unknown-type mixture problem, because the mixture objective is a weighted sum of two MNL revenue ratios and the revenue-ordered property does not generally apply.

In [18]:
def optimal_assortment_single_type(df_hotels, v, price_col=PRICE_COL):
    df = df_hotels.copy()
    df["_original_index"] = np.arange(len(df))
    df["_v"] = v

    # Revenue-ordered assortment property for single-type MNL.
    df = df.sort_values(price_col, ascending=False).reset_index(drop=True)

    numerator = 0.0
    denominator = 1.0
    chosen_original_indices = []

    for _, row in df.iterrows():
        price = row[price_col]
        weight = row["_v"]

        current_revenue = numerator / denominator

        if price > current_revenue:
            numerator += price * weight
            denominator += weight
            chosen_original_indices.append(int(row["_original_index"]))
        else:
            break

    selected = np.zeros(len(df_hotels), dtype=int)
    selected[chosen_original_indices] = 1

    optimal_revenue = numerator / denominator

    return selected, optimal_revenue

### MILP formulation

The original decision variable is:

$$
x_j =
\begin{cases}
1 & \text{if hotel } j \text{ is displayed} \\
0 & \text{otherwise}
\end{cases}
$$

For the unknown-type case, the original objective is:

$$
\max_x \; R_{mix}(x)
=
\theta_1
\frac{\sum_j x_j r_j v_{j1}}
{1 + \sum_j x_j v_{j1}}
+
\theta_2
\frac{\sum_j x_j r_j v_{j2}}
{1 + \sum_j x_j v_{j2}}
$$

where:

- $r_j$ is the revenue from hotel $j$, here `price_usd`
- $v_{jk}$ is the preference weight of hotel $j$ for customer type $k$
- $\theta_k$ is the probability that the customer belongs to type $k$

For the known-type cases, the objectives are:

$$
\max_x \; R_1(x)
=
\frac{\sum_j x_j r_j v_{j1}}
{1 + \sum_j x_j v_{j1}}
$$

and

$$
\max_x \; R_2(x)
=
\frac{\sum_j x_j r_j v_{j2}}
{1 + \sum_j x_j v_{j2}}
$$

These objectives are nonlinear because they contain ratios. To convert the problem into a mixed-integer linear program, define:

$$
z_k =
\frac{1}{1 + \sum_j x_j v_{jk}}
$$

and

$$
y_{jk} = x_j z_k
$$

Using these definitions, the denominator relationship becomes:

$$
z_k + \sum_j v_{jk} y_{jk} = 1
$$

The expected revenue for type $k$ becomes:

$$
R_k(x) =
\sum_j r_j v_{jk} y_{jk}
$$

Therefore, the linearized objective for the unknown-type case is:

$$
\max \;
\theta_1 \sum_j r_j v_{j1} y_{j1}
+
\theta_2 \sum_j r_j v_{j2} y_{j2}
$$

Or:

$$
\max \;
\sum_k \theta_k \sum_j r_j v_{jk} y_{jk}
$$

Finally, we need constraints to force:

$$
y_{jk} = x_j z_k
$$

Because $x_j$ is binary, this can be done with the following linear constraints:

$$
y_{jk} \le z_k
$$

$$
y_{jk} \le x_j
$$

$$
y_{jk} \ge z_k - (1 - x_j)
$$

Together with:

$$
x_j \in \{0,1\}
$$

and:

$$
0 \le z_k \le 1, \qquad 0 \le y_{jk} \le 1
$$

this gives a mixed-integer linear program that OR-Tools can solve.

In [19]:

def solve_assortment_ortools(revenue, v_by_type, theta_by_type, time_limit_seconds=300):
    revenue = np.asarray(revenue, dtype=float)
    v_by_type = [np.asarray(v, dtype=float) for v in v_by_type]

    n = len(revenue)
    K = len(v_by_type)

    solver = pywraplp.Solver.CreateSolver("CBC")
    if solver is None:
        raise RuntimeError("CBC solver is not available. Try reinstalling OR-Tools.")

    solver.SetTimeLimit(int(time_limit_seconds * 1000))

    hotels = range(n)
    types = range(K)

    # x[j] = 1 if hotel j is displayed.
    x = {j: solver.BoolVar(f"x_{j}") for j in hotels}

    # z[k] = inverse denominator for customer type k.
    z = {k: solver.NumVar(0.0, 1.0, f"z_{k}") for k in types}

    # y[j,k] = x[j] * z[k].
    y = {
        (j, k): solver.NumVar(0.0, 1.0, f"y_{j}_{k}")
        for j in hotels
        for k in types
    }

    # z[k] + sum_j v[j,k] y[j,k] = 1
    for k in types:
        solver.Add(
            z[k] + solver.Sum(v_by_type[k][j] * y[j, k] for j in hotels) == 1
        )

    # Linearization constraints for y[j,k] = x[j] * z[k]
    for j in hotels:
        for k in types:
            solver.Add(y[j, k] <= z[k])
            solver.Add(y[j, k] <= x[j])
            solver.Add(y[j, k] >= z[k] - (1 - x[j]))

    # Maximize expected revenue.
    objective = solver.Sum(
        theta_by_type[k] * revenue[j] * v_by_type[k][j] * y[j, k]
        for k in types
        for j in hotels
    )

    solver.Maximize(objective)

    status = solver.Solve()

    if status not in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        print("WARNING: Solver did not find an optimal or feasible solution.")
        print("Status:", status)

    selected = np.array([int(round(x[j].solution_value())) for j in hotels])
    objective_value = solver.Objective().Value()

    return selected, objective_value, status

### Solve for one dataset

For each dataset, we compute:

- `S`: optimal assortment under the unknown-type mixture model, using OR-Tools MILP.
- `S1`: optimal assortment if the customer is known to be type 1, using the single-type MNL greedy rule.
- `S2`: optimal assortment if the customer is known to be type 2, using the single-type MNL greedy rule.

Then we compare:

- revenue of `S` vs. `S1` under type 1
- revenue of `S` vs. `S2` under type 2

In [20]:
def interpret_status(status):
    if status == pywraplp.Solver.OPTIMAL:
        return "OPTIMAL"
    elif status == pywraplp.Solver.FEASIBLE:
        return "FEASIBLE"
    elif status == pywraplp.Solver.INFEASIBLE:
        return "INFEASIBLE"
    elif status == pywraplp.Solver.UNBOUNDED:
        return "UNBOUNDED"
    else:
        return f"OTHER ({status})"


def solve_problem5_for_dataset(file_path):
    df_hotels = pd.read_csv(file_path).copy()

    revenue = df_hotels[PRICE_COL].to_numpy(dtype=float)

    v1 = compute_preference_weights(df_hotels, beta_early)
    v2 = compute_preference_weights(df_hotels, beta_late)

    # S: unknown customer type.
    # This is the true mixture problem, so we use the MILP.
    S, mix_obj, status_mix = solve_assortment_ortools(
        revenue=revenue,
        v_by_type=[v1, v2],
        theta_by_type=[theta_1, theta_2],
    )

    # S1: known type 1.
    # This is a standard single-type MNL assortment problem,
    # so the revenue-ordered greedy solution is exact.
    S1, type1_obj = optimal_assortment_single_type(
        df_hotels=df_hotels,
        v=v1,
        price_col=PRICE_COL,
    )

    # S2: known type 2.
    # Same logic as S1.
    S2, type2_obj = optimal_assortment_single_type(
        df_hotels=df_hotels,
        v=v2,
        price_col=PRICE_COL,
    )

    # Comparisons requested in the problem.
    rev_S_under_type1 = expected_revenue_type(S, revenue, v1)
    rev_S1_under_type1 = expected_revenue_type(S1, revenue, v1)

    rev_S_under_type2 = expected_revenue_type(S, revenue, v2)
    rev_S2_under_type2 = expected_revenue_type(S2, revenue, v2)

    summary = {
        "dataset": file_path,
        "n_hotels": len(df_hotels),

        "S_size": int(S.sum()),
        "S1_size": int(S1.sum()),
        "S2_size": int(S2.sum()),

        "mixture_revenue_of_S": mix_obj,

        "revenue_S_under_type1": rev_S_under_type1,
        "revenue_S1_under_type1": rev_S1_under_type1,
        "type1_gain_from_personalization": rev_S1_under_type1 - rev_S_under_type1,

        "revenue_S_under_type2": rev_S_under_type2,
        "revenue_S2_under_type2": rev_S2_under_type2,
        "type2_gain_from_personalization": rev_S2_under_type2 - rev_S_under_type2,

        "status_mix": interpret_status(status_mix),
        "method_S": "OR-Tools MILP",
        "method_S1": "Revenue-ordered greedy",
        "method_S2": "Revenue-ordered greedy",
    }

    selected_hotels = {
        "S_unknown_type": df_hotels.loc[S == 1].copy(),
        "S1_type1": df_hotels.loc[S1 == 1].copy(),
        "S2_type2": df_hotels.loc[S2 == 1].copy(),
    }

    return summary, selected_hotels

### Run for all four datasets

In [21]:
dataset_files = ["data1.csv", "data2.csv", "data3.csv", "data4.csv"]

all_results = []
all_selected_hotels = {}

for file_path in dataset_files:
    print(f"Solving {file_path}...")
    result_summary, selected_hotels = solve_problem5_for_dataset(file_path)

    all_results.append(result_summary)
    all_selected_hotels[file_path] = selected_hotels

results_df = pd.DataFrame(all_results)
results_df

Solving data1.csv...
Solving data2.csv...
Solving data3.csv...
Solving data4.csv...


,dataset,n_hotels,S_size,S1_size,S2_size,mixture_revenue_of_S,revenue_S_under_type1,revenue_S1_under_type1,type1_gain_from_personalization,revenue_S_under_type2,revenue_S2_under_type2,type2_gain_from_personalization,status_mix,method_S,method_S1,method_S2
0,data1.csv,27,18,21,16,107.331551,103.248060,103.466261,0.218201,112.185309,112.295189,0.109880,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy
1,data2.csv,29,10,10,7,131.248269,126.928700,126.928700,0.000000,136.382637,137.261881,0.879243,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy
2,data3.csv,26,18,19,17,121.027995,117.427182,117.510103,0.082922,125.308029,125.431845,0.123816,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy
3,data4.csv,27,11,11,10,97.355438,94.684094,94.684094,0.000000,100.530677,100.587919,0.057242,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy


### Let's inspect the optimal assortments.

In [22]:
for file_path in dataset_files:
    print("\n", file_path)
    print("S  unknown type:", list(all_selected_hotels[file_path]["S_unknown_type"].index))
    print("S1 type 1:", list(all_selected_hotels[file_path]["S1_type1"].index))
    print("S2 type 2:", list(all_selected_hotels[file_path]["S2_type2"].index))




 data1.csv
S  unknown type: [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
S1 type 1: [0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26]
S2 type 2: [0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24]

 data2.csv
S  unknown type: [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
S1 type 1: [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
S2 type 2: [0, 1, 6, 7, 8, 10, 21]

 data3.csv
S  unknown type: [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
S1 type 1: [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24, 25]
S2 type 2: [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 19, 23, 24]

 data4.csv
S  unknown type: [3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26]
S1 type 1: [3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26]
S2 type 2: [3, 4, 6, 10, 15, 18, 19, 20, 21, 26]


### Export results

In [23]:
results_df.to_csv("problem5_summary_results.csv", index=False)

for file_path in dataset_files:
    base = file_path.replace(".csv", "")
    all_selected_hotels[file_path]["S_unknown_type"].to_csv(f"{base}_S_unknown_type.csv", index=False)
    all_selected_hotels[file_path]["S1_type1"].to_csv(f"{base}_S1_type1.csv", index=False)
    all_selected_hotels[file_path]["S2_type2"].to_csv(f"{base}_S2_type2.csv", index=False)

print("Saved summary and selected-hotel CSV files.")

Saved summary and selected-hotel CSV files.


## Problem 5: Early vs. Late Reservations

We computed three assortments:

- **S**: optimal when the customer type was unknown (mixture model)  
- **S1**: optimal for early customers  
- **S2**: optimal for late customers  

### Results

When the type was unknown, the assortment **S** reflected a compromise between both segments. Across datasets, we observed that **S** differed in size and composition from both **S1** and **S2**. For example, in *data1.csv*, S included 18 hotels, compared to 21 in S1 and 16 in S2.

When we assumed the customer type was known, the tailored assortments **S1** and **S2** consistently performed at least as well as S, and often slightly better.

For **early customers (type 1)**:
- In *data1.csv*, revenue increased from **103.25 to 103.47** (+0.22)
- In *data3.csv*, from **117.43 to 117.51** (+0.08)
- In *data2.csv* and *data4.csv*, there was **no gain** (same revenue)

For **late customers (type 2)**:
- In *data1.csv*, revenue increased from **112.19 to 112.30** (+0.11)
- In *data2.csv*, from **136.38 to 137.26** (+0.88)
- In *data3.csv*, from **125.31 to 125.43** (+0.12)
- In *data4.csv*, from **100.53 to 100.59** (+0.06)

### Key comparison

- Under type 1: **S1 performed as well as or slightly better than S**  
- Under type 2: **S2 consistently performed better than S**

### Takeaway

We found that knowing the customer type allowed us to slightly improve revenue by tailoring the assortment. The gains were not large, but they were consistent, especially for late customers. Overall, the mixture solution worked well when the type was unknown, but simple segmentation (early vs. late booking) still provided measurable improvements.